# RFM Customer Segmentation

This notebook performs customer segmentation using the RFM framework, which groups customers based on their purchasing behavior.  
RFM stands for **Recency**, **Frequency**, and **Monetary value**, three widely used metrics in customer analytics.

Using the customer-level dataset created in the previous notebook, each customer is assigned an RFM score and grouped into meaningful behavioral segments such as Champions, Loyal Customers, and High Value At Risk.  

These segments help identify high-value customers, potential churn risks, and opportunities for targeted retention or marketing strategies.

This cell imports the necessary Python libraries and loads the processed customer base table created in the previous notebook.

The customer base table contains one row per customer with aggregated features such as purchase history, total spend, and recency. This dataset serves as the input for calculating the RFM metrics used in segmentation.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)

DATA_DIR = Path("../data/processed")

customer_base = pd.read_csv(DATA_DIR / "customer_base.csv")

customer_base.head(), customer_base.shape

This cell prepares the dataset for RFM analysis by selecting the three key behavioral metrics: recency, frequency, and monetary value.

The columns are also renamed to match standard RFM terminology, which improves clarity and aligns the analysis with commonly used customer segmentation frameworks.

In [ ]:
rfm = customer_base[[
    "customer_unique_id",
    "recency_days",
    "n_orders",
    "total_spend"
]].copy()

rfm.rename(columns={
    "n_orders": "frequency",
    "total_spend": "monetary"
}, inplace=True)

rfm.describe()

This cell calculates the individual R, F, and M scores for each customer using quantile-based binning.

Customers are divided into five equally sized groups for each metric. For recency, lower values indicate more recent activity, so the scoring is reversed to ensure that higher scores represent more valuable customer behavior. Frequency and monetary values are scored in ascending order so that higher spending and purchase activity receive higher scores.

In [ ]:
rfm["R_score"] = pd.qcut(rfm["recency_days"], q=5, labels=[5,4,3,2,1]).astype(int)
rfm["F_score"] = pd.qcut(rfm["frequency"].rank(method="first"), q=5, labels=[1,2,3,4,5]).astype(int)
rfm["M_score"] = pd.qcut(rfm["monetary"], q=5, labels=[1,2,3,4,5]).astype(int)

rfm.head()

This cell combines the individual R, F, and M scores into a single three-digit RFM score for each customer.

The combined score provides a compact representation of customer behavior and allows similar purchasing patterns to be easily identified across the customer base.

In [ ]:
rfm["RFM_score"] = (
    rfm["R_score"].astype(str) +
    rfm["F_score"].astype(str) +
    rfm["M_score"].astype(str)
)

rfm["RFM_score"].value_counts().head(10)

This cell assigns each customer to a behavioral segment based on their RFM scores.

The segmentation rules group customers into categories such as Champions, Loyal Customers, High Value At Risk, New Customers, and Hibernating customers. These segments capture different patterns of engagement and spending, making it easier to identify high-value customers as well as customers who may require retention efforts.

In [ ]:
def rfm_segment(row):
    if row["R_score"] >= 4 and row["F_score"] >= 4 and row["M_score"] >= 4:
        return "Champions"
    if row["R_score"] >= 3 and row["F_score"] >= 3:
        return "Loyal Customers"
    if row["R_score"] <= 2 and row["M_score"] >= 4:
        return "High Value At Risk"
    if row["R_score"] >= 4 and row["F_score"] == 1:
        return "New Customers"
    if row["R_score"] <= 2 and row["F_score"] == 1:
        return "Hibernating"
    return "Others"

rfm["segment"] = rfm.apply(rfm_segment, axis=1)

rfm["segment"].value_counts()

This cell generates a summary profile for each customer segment.

The summary includes the number of customers, average recency, average purchase frequency, average spending, and the total revenue contributed by each segment. This helps quantify the business value of each segment and highlights which groups contribute most to overall revenue.

In [ ]:
segment_profile = (
    rfm.groupby("segment")
    .agg(
        customers=("customer_unique_id", "count"),
        avg_recency=("recency_days", "mean"),
        avg_frequency=("frequency", "mean"),
        avg_monetary=("monetary", "mean"),
        total_revenue=("monetary", "sum"),
    )
    .sort_values("total_revenue", ascending=False)
)

segment_profile

This cell saves the segmentation results to a processed CSV file.

Only the essential fields — customer identifier, assigned segment, and RFM score — are stored so that later notebooks can easily load the segmentation results for churn analysis and customer value modeling.

In [ ]:
out_path = Path("../data/processed/customer_segments.csv")
rfm[["customer_unique_id", "segment", "RFM_score"]].to_csv(out_path, index=False)

print("Saved:", out_path)

This cell prints the final distribution of customers across segments and displays the segment summary table.

These outputs serve as a final validation step to confirm that the segmentation logic produced reasonable customer groupings before proceeding to the next stage of the analysis.

In [ ]:
print("Customer count per segment:")
print(rfm["segment"].value_counts())

print("\nSegment profile summary:")
display(segment_profile)